# Perfilamiento de la fuente API: Evaluaciones Agropecuarias Municipales (EVA)

Fuente secundaria de la Segunda Entrega (R6 y R7). Se observa la fuente tal como la entrega la API,
Cada hallazgo se resume en `notebooks/hallazgos_perfilamiento.md`.

- Productor: UPRA (Unidad de Planificación Rural Agropecuaria), publicado en datos.gov.co (Socrata).
- Dataset: `uejq-wxrr`. Años de interés: 2024 y 2025.

In [1]:
# Configuracion. Las rutas se resuelven desde la raiz del repositorio para que el
# notebook funcione igual en VS Code, Jupyter o nbconvert.
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

URL_DATOS = 'https://www.datos.gov.co/resource/uejq-wxrr.json'
URL_METADATOS = 'https://www.datos.gov.co/api/views/uejq-wxrr.json'
ANIOS = (2024, 2025)
TAM_PAGINA = 5000
TIMEOUT_S = 60

# Copia local de la extraccion para no consultar la API en cada ejecucion.
# data/incoming/* esta ignorado por git: esta copia no se versiona.
RUTA_COPIA = RAIZ / 'data' / 'incoming' / 'profiling' / 'eva_2024_2025.json'
FORZAR_EXTRACCION = False

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
print('Raiz del proyecto encontrada:', (RAIZ / 'docker-compose.yaml').exists())

Raiz del proyecto encontrada: True


In [2]:
# 1. Metadatos publicados de la fuente: titulo, fecha de ultima actualizacion y
# tipo declarado de cada columna. Se comparan despues con lo que llega realmente.
meta = requests.get(URL_METADATOS, timeout=TIMEOUT_S).json()
print('Titulo:', meta.get('name'))
print('Atribucion:', meta.get('attribution'))
for campo in ('createdAt', 'rowsUpdatedAt', 'viewLastModified'):
    if meta.get(campo):
        print(f'{campo}:', datetime.fromtimestamp(meta[campo], tz=timezone.utc).isoformat())
print('Licencia:', (meta.get('license') or {}).get('name'))

columnas_meta = pd.DataFrame([
    {'campo_api': c['fieldName'], 'nombre': c['name'], 'tipo_declarado': c['dataTypeName'],
     'descripcion': (c.get('description') or '')[:90]}
    for c in meta['columns'] if not c['fieldName'].startswith(':')
])
columnas_meta

Titulo: Evaluaciones Agropecuarias Municipales – EVA. 2019 - 2025. Base Agrícola
Atribucion: Unidad de Planificación de Tierras Rurales, Adecuación de Tierras y Usos Agropecuarios - UPRA, Bogotá D.C.
createdAt: 2022-05-04T17:00:34+00:00
rowsUpdatedAt: 2026-08-28T16:54:03+00:00
viewLastModified: 2026-08-28T16:52:59+00:00
Licencia: Creative Commons Attribution | Share Alike 4.0 International


,campo_api,nombre,tipo_declarado,descripcion
0,c_digo_dane_departamento,Código Dane departamento,text,Código del departamento - DIVIPOLA
1,departamento,Departamento,text,Nombre del departamento
2,c_digo_dane_municipio,Código Dane municipio,text,Código del municipio - DIVIPOLA
3,municipio,Municipio,text,Nombre del municipio
4,grupo_cultivo,Grupo cultivo,text,Clasificación de la especie cultivada
5,subgrupo,Subgrupo,text,Subgrupo de la clasificación de especie cultivada
6,cultivo,Cultivo,text,Nombre de la especie cultivada
7,desagregaci_n_cultivo,Desagregación cultivo,text,"Desagregación de la especie, reportada por los..."
8,a_o,Año,number,Año de referencia
9,periodo,Periodo,text,Periodo de siembra (Año y semestre para transi...


In [3]:
# 2. Conteo de referencia por anio, calculado por la propia API. Sirve para
# verificar despues que la extraccion paginada no perdio ni duplico registros.
def contar_api(where):
    r = requests.get(URL_DATOS, params={'$select': 'count(*) AS n', '$where': where}, timeout=TIMEOUT_S)
    r.raise_for_status()
    return int(r.json()[0]['n'])

conteo_api = {anio: contar_api(f'a_o={anio}') for anio in ANIOS}
conteo_api

{2024: 25553, 2025: 25617}

In [4]:
# 3. Extraccion paginada. Se ordena por el identificador interno de Socrata (:id)
# para que las paginas sean estables y no se repitan ni salten registros.
def extraer_anio(anio):
    registros, offset, paginas = [], 0, []
    while True:
        params = {'$where': f'a_o={anio}', '$order': ':id', '$limit': TAM_PAGINA, '$offset': offset}
        r = requests.get(URL_DATOS, params=params, timeout=TIMEOUT_S)
        r.raise_for_status()
        pagina = r.json()
        paginas.append(len(pagina))
        registros.extend(pagina)
        if len(pagina) < TAM_PAGINA:
            break
        offset += TAM_PAGINA
    return registros, paginas

if RUTA_COPIA.exists() and not FORZAR_EXTRACCION:
    copia = json.loads(RUTA_COPIA.read_text(encoding='utf-8'))
    print('Usando copia local del', copia['fecha_extraccion_utc'])
else:
    copia = {'fecha_extraccion_utc': datetime.now(timezone.utc).isoformat(),
             'url': URL_DATOS, 'tam_pagina': TAM_PAGINA, 'paginas': {}, 'registros': []}
    for anio in ANIOS:
        regs, paginas = extraer_anio(anio)
        copia['paginas'][str(anio)] = paginas
        copia['registros'].extend(regs)
    RUTA_COPIA.parent.mkdir(parents=True, exist_ok=True)
    RUTA_COPIA.write_text(json.dumps(copia, ensure_ascii=False), encoding='utf-8')
    print('Extraccion nueva guardada en', RUTA_COPIA.relative_to(RAIZ))

print('Fecha de extraccion:', copia['fecha_extraccion_utc'])
print('Registros por pagina:', copia['paginas'])
print('Total registros extraidos:', len(copia['registros']))

Usando copia local del 2026-10-20T23:37:02.719850+00:00
Fecha de extraccion: 2026-10-20T23:37:02.719850+00:00
Registros por pagina: {'2024': [5000, 5000, 5000, 5000, 5000, 553], '2025': [5000, 5000, 5000, 5000, 5000, 617]}
Total registros extraidos: 51170


In [5]:
# 4. Conciliacion: registros extraidos por anio frente al conteo de la API.
# La API entrega todo como texto; no se convierte nada en este paso.
df = pd.DataFrame(copia['registros'])
extraidos = df['a_o'].value_counts().sort_index()
conciliacion = pd.DataFrame({'conteo_api': pd.Series({str(k): v for k, v in conteo_api.items()}),
                             'extraidos': extraidos})
conciliacion['diferencia'] = conciliacion['extraidos'] - conciliacion['conteo_api']
print('Forma del DataFrame:', df.shape)
conciliacion

Forma del DataFrame: (51170, 18)


,conteo_api,extraidos,diferencia
2024,25553,25553,0
2025,25617,25617,0


In [6]:
# 5. Esquema recibido. Socrata omite en el JSON los campos nulos de un registro,
# asi que un campo ausente aparece como NaN. Se mide la presencia de cada campo.
esquema = pd.DataFrame({
    'tipo_python': pd.Series({c: ', '.join(sorted({type(v).__name__ for v in df[c].dropna()})) for c in df.columns}),
    'presentes': df.notna().sum(),
    'ausentes': df.isna().sum(),
    'pct_ausentes': (df.isna().mean() * 100).round(3),
    'distintos': df.nunique(),
})
esquema = esquema.join(columnas_meta.set_index('campo_api')[['tipo_declarado']], how='left')
esquema

,tipo_python,presentes,ausentes,pct_ausentes,distintos,tipo_declarado
c_digo_dane_departamento,str,51170,0,0.0,32,text
departamento,str,51170,0,0.0,32,text
c_digo_dane_municipio,str,51170,0,0.0,1102,text
municipio,str,51170,0,0.0,1020,text
grupo_cultivo,str,51170,0,0.0,8,text
subgrupo,str,51170,0,0.0,23,text
cultivo,str,51170,0,0.0,160,text
desagregaci_n_cultivo,str,51170,0,0.0,187,text
a_o,str,51170,0,0.0,2,number
periodo,str,51170,0,0.0,6,text


In [7]:
# 6. Valores de texto problematicos: vacios, solo espacios, espacios al inicio o
# final y marcadores de 'sin dato'. Un campo presente no garantiza un dato util.
MARCADORES = {'', 'NA', 'N/A', 'ND', 'NULL', 'NONE', 'SIN DATO', 'SIN INFORMACION', 'NO APLICA', '-'}
texto = pd.DataFrame({
    'vacios_o_espacios': df.apply(lambda s: s.str.strip().eq('').sum()),
    'espacios_borde': df.apply(lambda s: (s != s.str.strip()).sum()),
    'marcadores_sin_dato': df.apply(lambda s: s.str.strip().str.upper().isin(MARCADORES).sum()),
})
print('Valores de cada marcador encontrados:')
for col in df.columns:
    encontrados = df[col][df[col].str.strip().str.upper().isin(MARCADORES)].value_counts()
    if len(encontrados):
        print(f'  {col}: {dict(encontrados)}')
texto[texto.sum(axis=1) > 0]

Valores de cada marcador encontrados:


  nombre_cient_fico_del_cultivo: {'No aplica': np.int64(815)}


,vacios_o_espacios,espacios_borde,marcadores_sin_dato
nombre_cient_fico_del_cultivo,0,0,815


In [8]:
# 7. Campos numericos: la API los declara 'number' pero llegan como texto.
# Se revisa el formato (solo digitos y punto decimal) antes de convertir, y se
# convierte en una copia para describir rangos, ceros y negativos.
NUMERICOS = ['a_o', 'rea_sembrada', 'rea_cosechada', 'producci_n', 'rendimiento', 'c_digo_del_cultivo']
PATRON_NUMERO = r'^-?\d+(\.\d+)?$'
formato = pd.DataFrame({
    'no_cumple_patron': {c: (~df[c].str.match(PATRON_NUMERO)).sum() for c in NUMERICOS},
    'con_coma': {c: df[c].str.contains(',').sum() for c in NUMERICOS},
    'decimales_max': {c: df[c].str.extract(r'\.(\d+)$')[0].str.len().max() for c in NUMERICOS},
})
print(formato, '\n')

num = df[NUMERICOS].apply(pd.to_numeric, errors='coerce')
num['anio'] = df['a_o']
resumen = []
for c in ['rea_sembrada', 'rea_cosechada', 'producci_n', 'rendimiento']:
    for anio, s in num.groupby('anio')[c]:
        resumen.append({'campo': c, 'anio': anio, 'no_convertibles': s.isna().sum(), 'negativos': (s < 0).sum(),
                        'ceros': (s == 0).sum(), 'pct_ceros': round((s == 0).mean() * 100, 2),
                        'min': s.min(), 'mediana': s.median(), 'p99': s.quantile(0.99), 'max': s.max()})
pd.DataFrame(resumen)

                    no_cumple_patron  con_coma  decimales_max
a_o                                0         0            NaN
rea_sembrada                       0         0            2.0
rea_cosechada                      0         0            2.0
producci_n                         0         0            2.0
rendimiento                        0         0            2.0
c_digo_del_cultivo                 0         0            NaN 



,campo,anio,no_convertibles,negativos,ceros,pct_ceros,min,mediana,p99,max
0,rea_sembrada,2024,0,0,705,2.76,0.0,17.00,3403.3600,57000.0
1,rea_sembrada,2025,0,0,710,2.77,0.0,17.00,3499.0736,77630.0
2,rea_cosechada,2024,0,0,1850,7.24,0.0,14.72,3105.4400,57000.0
3,rea_cosechada,2025,0,0,1686,6.58,0.0,15.00,3199.0400,77630.0
4,producci_n,2024,0,0,1850,7.24,0.0,87.50,35580.4856,4746000.0
5,producci_n,2025,0,0,1686,6.58,0.0,90.00,38684.9280,4758600.0
6,rendimiento,2024,0,0,1850,7.24,0.0,6.00,85.0000,253.0
7,rendimiento,2025,0,0,1686,6.58,0.0,6.00,86.0000,200.0


In [9]:
# 8. Codigos territoriales: longitud, relacion municipio -> departamento y
# correspondencia 1:1 entre codigo y nombre.
print('Longitud codigo departamento:', df['c_digo_dane_departamento'].str.len().value_counts().to_dict())
print('Longitud codigo municipio:', df['c_digo_dane_municipio'].str.len().value_counts().to_dict())
prefijo_ok = df['c_digo_dane_municipio'].str[:2] == df['c_digo_dane_departamento']
print('Municipios cuyo prefijo no coincide con su departamento:', (~prefijo_ok).sum())

nombres_por_codigo_dep = df.groupby('c_digo_dane_departamento')['departamento'].nunique()
codigos_por_nombre_dep = df.groupby('departamento')['c_digo_dane_departamento'].nunique()
print('Departamentos con mas de un nombre por codigo:', (nombres_por_codigo_dep > 1).sum())
print('Nombres de departamento con mas de un codigo:', (codigos_por_nombre_dep > 1).sum())

nombres_por_codigo_mun = df.groupby('c_digo_dane_municipio')['municipio'].nunique()
print('Municipios con mas de un nombre por codigo:', (nombres_por_codigo_mun > 1).sum())
if (nombres_por_codigo_mun > 1).any():
    print(df[df['c_digo_dane_municipio'].isin(nombres_por_codigo_mun[nombres_por_codigo_mun > 1].index)]
          .groupby('c_digo_dane_municipio')['municipio'].unique().head(10))

Longitud codigo departamento: {2: 51170}
Longitud codigo municipio: {5: 51170}
Municipios cuyo prefijo no coincide con su departamento: 0
Departamentos con mas de un nombre por codigo: 0
Nombres de departamento con mas de un codigo: 0
Municipios con mas de un nombre por codigo: 0


In [10]:
# 9. Cruce con la referencia DIVIPOLA de la Primera Entrega (33 departamentos
# presentes en la ECV 2025). Mientras se porta al nuevo repositorio, se lee
# desde el repositorio de la Primera Entrega clonado al lado.
RUTA_DIVIPOLA = RAIZ / 'data' / 'reference' / 'divipola_departamentos.csv'
divipola = pd.read_csv(RUTA_DIVIPOLA, dtype=str)
cod_ref = set(divipola['Código Departamento'])
cod_eva = set(df['c_digo_dane_departamento'])
print('Departamentos en referencia:', len(cod_ref), '| en EVA:', len(cod_eva))
print('En referencia y no en EVA:', sorted(cod_ref - cod_eva),
      divipola[divipola['Código Departamento'].isin(cod_ref - cod_eva)]['Nombre Departamento'].tolist())
print('En EVA y no en referencia:', sorted(cod_eva - cod_ref))

# Nombres: EVA usa mayusculas/minusculas y tildes distintas a DIVIPOLA. Se
# compara solo para documentar; la integracion se hara por codigo.
nombres = (df[['c_digo_dane_departamento', 'departamento']].drop_duplicates()
           .merge(divipola, left_on='c_digo_dane_departamento', right_on='Código Departamento', how='left'))
diferentes = nombres[nombres['departamento'].str.upper() != nombres['Nombre Departamento'].str.upper()]
print('Departamentos con nombre distinto al de DIVIPOLA (ignorando mayusculas):', len(diferentes))
diferentes[['c_digo_dane_departamento', 'departamento', 'Nombre Departamento']]

Departamentos en referencia: 33 | en EVA: 32
En referencia y no en EVA: ['11'] ['BOGOTÁ, D.C.']
En EVA y no en referencia: []
Departamentos con nombre distinto al de DIVIPOLA (ignorando mayusculas): 0


,c_digo_dane_departamento,departamento,Nombre Departamento


In [11]:
# 10. Dominios de las variables categoricas, por anio.
for col in ['grupo_cultivo', 'ciclo_del_cultivo', 'estado_f_sico_del_cultivo', 'periodo']:
    print(f'=== {col} ===')
    print(pd.crosstab(df[col], df['a_o'], margins=True, margins_name='total'), '\n')

=== grupo_cultivo ===
a_o                                                  2024   2025  total
grupo_cultivo                                                          
Cereales                                             4481   4459   8940
Cultivos para condimentos, bebidas medicinales ...    248    242    490
Cultivos tropicales tradicionales                    2105   2093   4198
Frutales                                             7597   7586  15183
Hortalizas                                           4909   5012   9921
Leguminosas                                          2646   2648   5294
Oleaginosas                                           302    311    613
Raíces y tubérculos                                  3265   3266   6531
total                                               25553  25617  51170 

=== ciclo_del_cultivo ===


a_o                 2024   2025  total
ciclo_del_cultivo                     
Permanente          9899   9879  19778
Transitorio        15654  15738  31392
total              25553  25617  51170 

=== estado_f_sico_del_cultivo ===
a_o                          2024   2025  total
estado_f_sico_del_cultivo                      
Aceite crudo                  141    142    283
Algodón semilla                58     53    111
Caña o verde                  677    673   1350
En fresco                   16825  16893  33718
Fibra o cabuya                 60     56    116
Grano                         646    633   1279
Grano o almendra              113    124    237
Grano seco                   5688   5692  11380
Paddy o cascara verde         697    700   1397
Pergamino o seco de trilla    648    651   1299
total                       25553  25617  51170 

=== periodo ===


a_o       2024   2025  total
periodo                     
2024      9955      0   9955
2024A     8013      0   8013
2024B     7585      0   7585
2025         0   9929   9929
2025A        0   8071   8071
2025B        0   7617   7617
total    25553  25617  51170 



In [12]:
# 11. Coherencia del periodo: debe empezar por el anio del registro, y el sufijo
# (A/B o ninguno) deberia corresponder al ciclo del cultivo.
print('Periodo que no empieza por el anio:', (~df.apply(lambda f: f['periodo'].startswith(f['a_o']), axis=1)).sum())
df_tmp = df.assign(tipo_periodo=df['periodo'].str[4:].replace('', 'anual'))
pd.crosstab(df_tmp['ciclo_del_cultivo'], df_tmp['tipo_periodo'])

Periodo que no empieza por el anio: 0


tipo_periodo,A,B,anual
ciclo_del_cultivo,,,
Permanente,0,0,19778
Transitorio,16084,15202,106


In [13]:
# 12. Jerarquia del cultivo. El codigo tiene 187 valores, igual que la
# desagregacion, mientras 'cultivo' tiene 160. Se verifica que nivel identifica el codigo.
def relacion(a, b):
    return {'max_b_por_a': int(df.groupby(a)[b].nunique().max()),
            'a_con_varios_b': int((df.groupby(a)[b].nunique() > 1).sum())}

relaciones = pd.DataFrame({
    'codigo -> desagregacion': relacion('c_digo_del_cultivo', 'desagregaci_n_cultivo'),
    'desagregacion -> codigo': relacion('desagregaci_n_cultivo', 'c_digo_del_cultivo'),
    'codigo -> cultivo': relacion('c_digo_del_cultivo', 'cultivo'),
    'cultivo -> codigo': relacion('cultivo', 'c_digo_del_cultivo'),
    'codigo -> estado fisico': relacion('c_digo_del_cultivo', 'estado_f_sico_del_cultivo'),
    'codigo -> ciclo': relacion('c_digo_del_cultivo', 'ciclo_del_cultivo'),
    'codigo -> grupo': relacion('c_digo_del_cultivo', 'grupo_cultivo'),
    'cultivo -> estado fisico': relacion('cultivo', 'estado_f_sico_del_cultivo'),
    'cultivo -> grupo': relacion('cultivo', 'grupo_cultivo'),
}).T
relaciones

,max_b_por_a,a_con_varios_b
codigo -> desagregacion,1,0
desagregacion -> codigo,1,0
codigo -> cultivo,1,0
cultivo -> codigo,5,16
codigo -> estado fisico,1,0
codigo -> ciclo,1,0
codigo -> grupo,1,0
cultivo -> estado fisico,1,0
cultivo -> grupo,1,0


In [14]:
# 13. Cultivos con varias desagregaciones (varios codigos). Es lo que hay que
# decidir al comparar entre anios: sumar a nivel cultivo o comparar por codigo.
varios = df.groupby('cultivo')['c_digo_del_cultivo'].nunique()
detalle = (df[df['cultivo'].isin(varios[varios > 1].index)]
           .groupby(['cultivo', 'c_digo_del_cultivo', 'desagregaci_n_cultivo', 'estado_f_sico_del_cultivo', 'ciclo_del_cultivo'])
           .size().rename('registros').reset_index())
print('Cultivos con mas de un codigo:', (varios > 1).sum())
pd.set_option('display.max_rows', 120)
detalle

Cultivos con mas de un codigo: 16


,cultivo,c_digo_del_cultivo,desagregaci_n_cultivo,estado_f_sico_del_cultivo,ciclo_del_cultivo,registros
0,Aguacate,2040201,Aguacate Hass,En fresco,Permanente,520
1,Aguacate,2040202,Aguacate Papelillo,En fresco,Permanente,216
2,Aguacate,2040299,Aguacate demás variedades,En fresco,Permanente,991
3,Arroz,1010101,Arroz riego,Paddy o cascara verde,Transitorio,461
4,Arroz,1010102,Arroz secano manual,Paddy o cascara verde,Transitorio,583
5,Arroz,1010103,Arroz secano mecanizado,Paddy o cascara verde,Transitorio,353
6,Banano,2040901,Banano de exportación,En fresco,Permanente,28
7,Banano,2040902,Banano consumo interno,En fresco,Permanente,568
8,Caña,2030401,Caña de azúcar,Caña o verde,Permanente,101
9,Caña,2030402,Caña panelera,Caña o verde,Permanente,1136


In [15]:
# 14. Unicidad: filas duplicadas exactas y busqueda de la llave natural del registro.
print('Filas duplicadas exactas:', df.duplicated().sum())
candidatas = {
    'municipio + codigo + periodo': ['c_digo_dane_municipio', 'c_digo_del_cultivo', 'periodo'],
    'municipio + cultivo + periodo': ['c_digo_dane_municipio', 'cultivo', 'periodo'],
    'municipio + codigo + periodo + estado': ['c_digo_dane_municipio', 'c_digo_del_cultivo', 'periodo', 'estado_f_sico_del_cultivo'],
    'municipio + desagregacion + periodo': ['c_digo_dane_municipio', 'desagregaci_n_cultivo', 'periodo'],
}
pd.DataFrame({nombre: {'filas_en_grupos_duplicados': int(df.duplicated(cols, keep=False).sum()),
                       'grupos_duplicados': int(df[df.duplicated(cols, keep=False)].groupby(cols).ngroups)}
              for nombre, cols in candidatas.items()}).T

Filas duplicadas exactas: 0


,filas_en_grupos_duplicados,grupos_duplicados
municipio + codigo + periodo,0,0
municipio + cultivo + periodo,10684,4757
municipio + codigo + periodo + estado,0,0
municipio + desagregacion + periodo,0,0


In [16]:
# 15. Ceros: produccion, area cosechada y rendimiento tienen el mismo numero de
# ceros. Se verifica si son las mismas filas y como se combinan con el area sembrada.
v = num.assign(cultivo=df['cultivo'], ciclo=df['ciclo_del_cultivo'])
patron = pd.DataFrame({
    'sembrada_0': v['rea_sembrada'] == 0, 'cosechada_0': v['rea_cosechada'] == 0,
    'produccion_0': v['producci_n'] == 0, 'rendimiento_0': v['rendimiento'] == 0,
})
print(patron.value_counts().rename('registros').to_frame(), '\n')
print('Produccion 0 por ciclo y anio:')
print(pd.crosstab(v['ciclo'][patron['produccion_0']], v['anio'][patron['produccion_0']]))

                                                   registros
sembrada_0 cosechada_0 produccion_0 rendimiento_0           
False      False       False        False              46222
           True        True         True                3533
True       False       False        False               1412
           True        True         True                   3 

Produccion 0 por ciclo y anio:
anio         2024  2025
ciclo                  
Permanente    336   261
Transitorio  1514  1425


In [17]:
# 16. Coherencia numerica entre campos.
# - El area cosechada no deberia superar el area sembrada.
# - Produccion positiva exige area cosechada positiva y viceversa.
# - Rendimiento declarado = produccion / area cosechada (redondeado a 2 decimales).
cos_mayor = v['rea_cosechada'] > v['rea_sembrada']
prod_sin_area = (v['producci_n'] > 0) & (v['rea_cosechada'] == 0)
area_sin_prod = (v['producci_n'] == 0) & (v['rea_cosechada'] > 0)
con_area = v['rea_cosechada'] > 0
rend_calc = (v['producci_n'] / v['rea_cosechada']).where(con_area)
dif_rel = ((v['rendimiento'] - rend_calc).abs() / rend_calc.where(rend_calc > 0))
print('Area cosechada > area sembrada:', cos_mayor.sum())
print('Produccion > 0 con area cosechada 0:', prod_sin_area.sum())
print('Area cosechada > 0 con produccion 0:', area_sin_prod.sum())
for tol in (0.01, 0.05, 0.10):
    print(f'Rendimiento difiere de produccion/area en mas de {tol:.0%}:', (dif_rel > tol).sum())
if cos_mayor.any():
    print('\nCasos con area cosechada > sembrada por ciclo:')
    print(v[cos_mayor].groupby(['ciclo', 'anio']).size())
    print(df.loc[cos_mayor, ['municipio', 'cultivo', 'periodo', 'rea_sembrada', 'rea_cosechada', 'producci_n']].head(8))

Area cosechada > area sembrada: 6950
Produccion > 0 con area cosechada 0: 0
Area cosechada > 0 con produccion 0: 0
Rendimiento difiere de produccion/area en mas de 1%: 15
Rendimiento difiere de produccion/area en mas de 5%: 0
Rendimiento difiere de produccion/area en mas de 10%: 0

Casos con area cosechada > sembrada por ciclo:
ciclo        anio
Permanente   2024       4
Transitorio  2024    3298
             2025    3648
dtype: int64
    municipio           cultivo periodo rea_sembrada rea_cosechada producci_n
16   Medellín            Frijol   2024A         9.50         14.00      18.20
26   Medellín  Otras hortalizas   2024A        34.00         36.00     252.00
28   Medellín              Papa   2024A        38.00         42.00     672.00
44   Medellín              Yuca   2024A         1.50          2.20      33.00
53  Abejorral            Frijol   2024A        80.00         85.00     127.50
68   Abriaquí            Arveja   2024B         1.04          1.50       1.50
72   Abriaquí  

In [18]:
# 17. Detalle del rendimiento inconsistente: magnitud de la diferencia y ejemplos.
inconsistentes = df.loc[dif_rel > 0.05, ['departamento', 'municipio', 'cultivo', 'periodo',
                                         'rea_cosechada', 'producci_n', 'rendimiento']].copy()
inconsistentes['rend_calculado'] = rend_calc[dif_rel > 0.05].round(2)
print('Registros con diferencia > 5%:', len(inconsistentes))
print(inconsistentes['cultivo'].value_counts().head(10))
inconsistentes.head(10)

Registros con diferencia > 5%: 0
Series([], Name: count, dtype: int64)


,departamento,municipio,cultivo,periodo,rea_cosechada,producci_n,rendimiento,rend_calculado


In [19]:
# 18. Transitorios con periodo anual (106 registros): de que cultivos son y si
# el mismo municipio-cultivo-anio tambien reporta semestres (riesgo de doble conteo).
trans_anual = df[(df['ciclo_del_cultivo'] == 'Transitorio') & (df['periodo'].str.len() == 4)]
print(trans_anual.groupby(['cultivo', 'a_o']).size().rename('registros'), '\n')

clave_anio = ['c_digo_dane_municipio', 'c_digo_del_cultivo', 'a_o']
tipos_por_clave = df.assign(semestral=df['periodo'].str.len() > 4).groupby(clave_anio)['semestral'].nunique()
print('Municipio-codigo-anio con periodo anual Y semestral a la vez:', (tipos_por_clave > 1).sum())

cultivo             a_o 
Plantas aromáticas  2024    56
                    2025    50
Name: registros, dtype: int64 



Municipio-codigo-anio con periodo anual Y semestral a la vez: 8


In [20]:
# 19. Valores atipicos. Mayores producciones (para validar que son plausibles) y
# rendimientos extremos relativos a la mediana del mismo codigo de cultivo.
top = df.assign(prod=v['producci_n']).nlargest(8, 'prod')
print(top[['departamento', 'municipio', 'desagregaci_n_cultivo', 'periodo', 'rea_cosechada', 'producci_n', 'rendimiento']], '\n')

med = v[v['rendimiento'] > 0].groupby(df['c_digo_del_cultivo'])['rendimiento'].median()
razon = v['rendimiento'] / df['c_digo_del_cultivo'].map(med)
for k in (5, 10, 20):
    print(f'Rendimiento > {k}x la mediana de su cultivo:', (razon > k).sum())
extremos = df.loc[razon > 10, ['departamento', 'municipio', 'desagregaci_n_cultivo', 'periodo', 'rea_cosechada', 'producci_n', 'rendimiento']]
extremos.assign(mediana_cultivo=df.loc[razon > 10, 'c_digo_del_cultivo'].map(med)).head(10)

          departamento   municipio desagregaci_n_cultivo periodo rea_cosechada  producci_n rendimiento
49509  Valle del Cauca     Palmira        Caña de azúcar    2025      33990.00  4758600.00      140.00
23874  Valle del Cauca     Palmira        Caña de azúcar    2024      33900.00  4746000.00      140.00
23351  Valle del Cauca  Candelaria        Caña de azúcar    2024      25391.00  3097702.00      122.00
48992  Valle del Cauca  Candelaria        Caña de azúcar    2025      25391.00  3097702.00      122.00
50135  Valle del Cauca      Zarzal        Caña de azúcar    2025      17670.00  2120400.00      120.00
24511  Valle del Cauca      Zarzal        Caña de azúcar    2024      17420.00  2090400.00      120.00
23502  Valle del Cauca  El Cerrito        Caña de azúcar    2024      16965.00  2035800.00      120.00
49143  Valle del Cauca  El Cerrito        Caña de azúcar    2025      16965.00  2035800.00      120.00 



Rendimiento > 5x la mediana de su cultivo: 29
Rendimiento > 10x la mediana de su cultivo: 2
Rendimiento > 20x la mediana de su cultivo: 0


,departamento,municipio,desagregaci_n_cultivo,periodo,rea_cosechada,producci_n,rendimiento,mediana_cultivo
1903,Antioquia,Sonsón,Agraz - mortiño,2024,3.00,90.00,30.00,2.0
27449,Antioquia,Sonsón,Agraz - mortiño,2025,3.00,90.00,30.00,2.0


In [21]:
# 20. Cobertura territorial por anio: municipios, codigos de cultivo y registros
# por departamento. Detecta departamentos con poca informacion agricola.
cob = (df.groupby(['c_digo_dane_departamento', 'departamento', 'a_o'])
         .agg(municipios=('c_digo_dane_municipio', 'nunique'), codigos=('c_digo_del_cultivo', 'nunique'),
              registros=('c_digo_del_cultivo', 'size'))
         .unstack('a_o'))
cob.columns = [f'{m}_{a}' for m, a in cob.columns]
print('Departamentos por anio:', df.groupby('a_o')['c_digo_dane_departamento'].nunique().to_dict())
print('Municipios por anio:', df.groupby('a_o')['c_digo_dane_municipio'].nunique().to_dict())
cob.sort_values('registros_2025')

Departamentos por anio: {'2024': 32, '2025': 32}
Municipios por anio: {'2024': 1101, '2025': 1102}


,,municipios_2024,municipios_2025,codigos_2024,codigos_2025,registros_2024,registros_2025
c_digo_dane_departamento,departamento,,,,,,
88,"Archipiélago de San Andrés, Providencia y Santa Catalina",2,2,21,21,36,37
94,Guainía,2,2,16,17,37,38
97,Vaupés,3,3,19,18,46,44
95,Guaviare,4,4,14,14,50,51
99,Vichada,4,4,23,22,60,63
91,Amazonas,2,2,32,31,70,70
81,Arauca,7,7,27,27,117,115
86,Putumayo,13,13,61,61,249,249
18,Caquetá,16,16,52,50,263,258


In [22]:
# 21. Municipios que aparecen solo en uno de los dos anios (cambian la base de comparacion).
mun_anio = df.groupby('c_digo_dane_municipio')['a_o'].unique().map(lambda a: ','.join(sorted(a)))
print(mun_anio.value_counts())
solo = mun_anio[mun_anio != '2024,2025']
df[df['c_digo_dane_municipio'].isin(solo.index)][['c_digo_dane_municipio', 'municipio', 'departamento', 'a_o']].drop_duplicates()

a_o
2024,2025    1101
2025            1
Name: count, dtype: int64


,c_digo_dane_municipio,municipio,departamento,a_o
40043,44560,Manaure,La Guajira,2025


In [23]:
# 22. Agregacion a la unidad de analisis de R6 y R7: departamento x codigo x anio,
# sumando municipios y periodos. Se valida que la agregacion no pierde produccion.
base = df[['c_digo_dane_departamento', 'cultivo', 'c_digo_del_cultivo', 'grupo_cultivo', 'a_o']].assign(
    produccion=v['producci_n'], area_cosechada=v['rea_cosechada'])
agg_cod = base.groupby(['c_digo_dane_departamento', 'cultivo', 'c_digo_del_cultivo', 'grupo_cultivo', 'a_o'],
                       as_index=False)[['produccion', 'area_cosechada']].sum()
print('Filas departamento-codigo-anio:', len(agg_cod))
print('Produccion total conservada:', round(base['produccion'].sum(), 2) == round(agg_cod['produccion'].sum(), 2))
print('Departamento-codigo-anio con produccion agregada = 0:', (agg_cod['produccion'] == 0).sum())

Filas departamento-codigo-anio: 3876
Produccion total conservada: True
Departamento-codigo-anio con produccion agregada = 0: 82


In [24]:
# 23. Comparabilidad 2024 vs 2025 (base de R7), a nivel codigo y a nivel cultivo.
def comparar(nivel):
    t = agg_cod.groupby(['c_digo_dane_departamento', nivel, 'a_o'])['produccion'].sum().unstack('a_o')
    clasif = pd.Series('comparable', index=t.index)
    clasif[t['2024'].isna()] = 'solo_2025'
    clasif[t['2025'].isna()] = 'solo_2024'
    clasif[(t['2024'] == 0) & t['2025'].notna()] = 'base_2024_cero'
    clasif[(clasif == 'comparable') & (t['2025'] == 0)] = 'comparable_2025_cero'
    return t, clasif

resultado = {}
for nivel in ('c_digo_del_cultivo', 'cultivo'):
    t, clasif = comparar(nivel)
    resultado[nivel] = clasif.value_counts()
print(pd.DataFrame(resultado).fillna(0).astype(int))

t_cul, clasif_cul = comparar('cultivo')
comparables = t_cul[clasif_cul.isin(['comparable', 'comparable_2025_cero'])]
var = (comparables['2025'] / comparables['2024'] - 1) * 100
print('\nVariacion % por departamento-cultivo (comparables):')
print(var.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(1))
print('Variaciones > +500%:', (var > 500).sum(), '| caidas del 100%:', (var == -100).sum())

                      c_digo_del_cultivo  cultivo
comparable                          1850     1527
base_2024_cero                        48       43
solo_2024                             41       34
solo_2025                             27       24
comparable_2025_cero                   6        3

Variacion % por departamento-cultivo (comparables):
count    1530.0
mean       11.9
std       134.1
min      -100.0
1%        -78.2
5%        -44.2
25%        -7.7
50%         0.0
75%        11.9
95%        67.5
99%       222.6
max      3142.9
dtype: float64
Variaciones > +500%: 7 | caidas del 100%: 3


In [25]:
# 24. Diversidad preliminar por departamento en 2025 (todos los grupos, sin filtrar
# todavia por cultivos alimentarios): numero de cultivos con produccion positiva.
div = (agg_cod[(agg_cod['a_o'] == '2025') & (agg_cod['produccion'] > 0)]
       .groupby('c_digo_dane_departamento').agg(cultivos=('cultivo', 'nunique'), codigos=('c_digo_del_cultivo', 'nunique')))
print(div.describe().round(1))
div.sort_values('cultivos').head(8)

       cultivos  codigos
count      32.0     32.0
mean       49.1     59.4
std        24.6     29.0
min        10.0     13.0
25%        34.8     44.2
50%        45.0     57.0
75%        61.5     75.0
max        97.0    109.0


,cultivos,codigos
c_digo_dane_departamento,,
95,10,13
94,15,17
97,17,18
81,19,27
99,20,21
88,20,21
91,29,31
70,34,45


In [26]:
# 25. Catalogo de cultivos para decidir la canasta alimentaria (decision del equipo).
# Se exporta completo a la carpeta de profiling (no versionada) y se resume por grupo.
catalogo = (df.assign(prod=v['producci_n'])
            .groupby(['grupo_cultivo', 'subgrupo', 'cultivo', 'estado_f_sico_del_cultivo', 'ciclo_del_cultivo'])
            .agg(codigos=('c_digo_del_cultivo', 'nunique'), registros=('cultivo', 'size'),
                 departamentos=('c_digo_dane_departamento', 'nunique'), produccion_2025_t=('prod', lambda s: s[df.loc[s.index, 'a_o'] == '2025'].sum()))
            .reset_index())
catalogo.to_csv(RUTA_COPIA.parent / 'catalogo_cultivos_eva.csv', index=False, encoding='utf-8-sig')
print('Cultivos distintos:', catalogo['cultivo'].nunique())
print(catalogo.groupby('grupo_cultivo').agg(cultivos=('cultivo', 'nunique'), subgrupos=('subgrupo', 'nunique'),
                                             produccion_2025_t=('produccion_2025_t', 'sum')).round(0))

Cultivos distintos: 160
                                                    cultivos  subgrupos  produccion_2025_t
grupo_cultivo                                                                             
Cereales                                                   8          1          5459678.0
Cultivos para condimentos, bebidas medicinales ...        25          4            18762.0
Cultivos tropicales tradicionales                          8          1         47966664.0
Frutales                                                  64          8         14208739.0
Hortalizas                                                31          6          3058076.0
Leguminosas                                                8          1           227934.0
Oleaginosas                                                5          1          2622640.0
Raíces y tubérculos                                       11          1          7785595.0


In [27]:
# 26. Definicion oficial completa de los campos de area, produccion y periodo.
# Necesaria para interpretar 'area cosechada > area sembrada' antes de llamarlo error.
for c in meta['columns']:
    if c['fieldName'] in ('periodo', 'rea_sembrada', 'rea_cosechada', 'producci_n', 'rendimiento', 'ciclo_del_cultivo'):
        print(f"{c['fieldName']}: {c.get('description')}")
print()
print('Descripcion del dataset:')
print(meta.get('description'))

periodo: Periodo de siembra (Año y semestre para transitorios) y solo año para los cultivos permanentes
rea_sembrada: Área Sembrada en hectáreas
rea_cosechada: Área cosechada en hectáreas de las siembras realizadas en el periodo de referencia
producci_n: Producción en toneladas de las siembras realizadas en el periodo de referencia
rendimiento: Rendimiento (producción / área cosechada)
ciclo_del_cultivo: Ciclo del cultivo (transitorio o permanente)

Descripcion del dataset:
La EVA es una investigación que se realiza mediante un mecanismo subjetivo de recolección de información respecto al área, la producción y el rendimiento de la actividad agropecuaria municipal. El método de cálculo se fundamenta principalmente en la confrontación y conciliación de la información que proveen los registros administrativos que manejan entidades y gremios del sector (área registrada, área bajo asistencia técnica, sacrificio de ganado, registros de parafiscales etc.), con el conocimiento de los funcionar

In [28]:
# 27. Los 8 casos municipio-codigo-anio con periodo anual y semestral a la vez.
dobles = tipos_por_clave[tipos_por_clave > 1].index
casos = df.set_index(clave_anio).loc[dobles].reset_index()
casos[['departamento', 'municipio', 'desagregaci_n_cultivo', 'ciclo_del_cultivo', 'a_o', 'periodo',
       'rea_sembrada', 'rea_cosechada', 'producci_n']].sort_values(['municipio', 'desagregaci_n_cultivo', 'periodo'])

,departamento,municipio,desagregaci_n_cultivo,ciclo_del_cultivo,a_o,periodo,rea_sembrada,rea_cosechada,producci_n
3,Cundinamarca,Chipaque,Plantas aromáticas,Transitorio,2024,2024,87.00,72.00,576.00
4,Cundinamarca,Chipaque,Plantas aromáticas,Transitorio,2024,2024A,13.00,13.00,78.00
5,Cundinamarca,Chipaque,Plantas aromáticas,Transitorio,2024,2024B,15.00,15.00,90.00
6,Cundinamarca,Chipaque,Plantas aromáticas,Transitorio,2025,2025,95.00,85.00,680.00
7,Cundinamarca,Chipaque,Plantas aromáticas,Transitorio,2025,2025A,20.00,20.00,120.00
8,Cundinamarca,Chipaque,Plantas aromáticas,Transitorio,2025,2025B,20.00,20.00,120.00
15,Tolima,Coello,Plantas aromáticas,Transitorio,2024,2024,5.00,5.00,2.50
16,Tolima,Coello,Plantas aromáticas,Transitorio,2024,2024A,5.00,5.00,42.50
17,Tolima,Coello,Plantas aromáticas,Transitorio,2024,2024B,5.00,5.00,42.50
18,Tolima,Coello,Plantas aromáticas,Transitorio,2025,2025,5.00,5.00,5.00


In [29]:
# 28. Valores repetidos entre anios. Se comparan los mismos municipio-codigo-tipo de
# periodo (anual, A, B) en 2024 y 2025. Una repeticion exacta de area y produccion
# puede indicar que la cifra se arrastro del anio anterior en lugar de medirse.
rep = df.assign(sufijo=df['periodo'].str[4:].replace('', 'anual'), prod=v['producci_n'], area=v['rea_cosechada'],
                sembrada=v['rea_sembrada'])
llave_rep = ['c_digo_dane_municipio', 'c_digo_del_cultivo', 'sufijo']
pares = rep[rep['a_o'] == '2024'].merge(rep[rep['a_o'] == '2025'], on=llave_rep, suffixes=('_24', '_25'))
pares = pares[pares['prod_24'] > 0]
identico_prod = pares['prod_24'] == pares['prod_25']
identico_todo = identico_prod & (pares['area_24'] == pares['area_25']) & (pares['sembrada_24'] == pares['sembrada_25'])
print('Pares municipio-codigo-periodo con produccion 2024 > 0 presentes en ambos anios:', len(pares))
print(f'Produccion identica 2024 = 2025: {identico_prod.sum()} ({identico_prod.mean():.1%})')
print(f'Area sembrada, cosechada y produccion identicas: {identico_todo.sum()} ({identico_todo.mean():.1%})')
print('\nPor ciclo:')
print(pares.assign(identico=identico_todo).groupby('ciclo_del_cultivo_24')['identico'].agg(['sum', 'mean']).round(3))

Pares municipio-codigo-periodo con produccion 2024 > 0 presentes en ambos anios: 23071
Produccion identica 2024 = 2025: 5923 (25.7%)
Area sembrada, cosechada y produccion identicas: 4373 (19.0%)

Por ciclo:
                       sum   mean
ciclo_del_cultivo_24             
Permanente            2575  0.275
Transitorio           1798  0.131


In [30]:
# 29. Repeticion exacta por departamento: participacion de pares identicos.
por_dep = (pares.assign(identico=identico_todo)
           .groupby(['c_digo_dane_departamento_24', 'departamento_24'])['identico'].agg(pares='size', identicos='sum', pct='mean'))
por_dep['pct'] = (por_dep['pct'] * 100).round(1)
por_dep.sort_values('pct', ascending=False)

,,pares,identicos,pct
c_digo_dane_departamento_24,departamento_24,,,
88,"Archipiélago de San Andrés, Providencia y Santa Catalina",30,12,40.0
63,Quindío,248,82,33.1
97,Vaupés,38,12,31.6
17,Caldas,556,169,30.4
76,Valle del Cauca,1487,430,28.9
47,Magdalena,586,169,28.8
08,Atlántico,359,90,25.1
20,Cesar,601,140,23.3
50,Meta,572,121,21.2


In [31]:
# 30. Efecto sobre R7: a nivel departamento-cultivo, cuantas variaciones son
# exactamente 0 % y que parte de la produccion comparable proviene de pares repetidos.
print('Variaciones departamento-cultivo exactamente 0 %:', (var == 0).sum(), 'de', len(var), f'({(var == 0).mean():.1%})')
prod_rep = pares.loc[identico_todo, 'prod_25'].sum() / pares['prod_25'].sum()
print(f'Parte de la produccion 2025 comparable que repite exactamente 2024: {prod_rep:.1%}')

Variaciones departamento-cultivo exactamente 0 %: 181 de 1530 (11.8%)
Parte de la produccion 2025 comparable que repite exactamente 2024: 24.5%


In [32]:
# 31. Catalogo completo por subgrupo (sin recortar) para decidir la canasta alimentaria.
for (grupo, subgrupo), g in catalogo.groupby(['grupo_cultivo', 'subgrupo']):
    print(f'## {grupo} / {subgrupo}')
    for _, f in g.sort_values('produccion_2025_t', ascending=False).iterrows():
        print(f"   {f['cultivo']:<45} {f['estado_f_sico_del_cultivo']:<28} deptos={f['departamentos']:>2}  prod_2025_t={f['produccion_2025_t']:>13,.0f}")

## Cereales / Cereales
   Arroz                                         Paddy o cascara verde        deptos=26  prod_2025_t=    3,774,140
   Maíz                                          Grano seco                   deptos=32  prod_2025_t=    1,675,380
   Cebada                                        En fresco                    deptos= 3  prod_2025_t=        3,340
   Trigo                                         En fresco                    deptos= 5  prod_2025_t=        2,173
   Sorgo                                         En fresco                    deptos=10  prod_2025_t=        1,896
   Avena                                         En fresco                    deptos= 3  prod_2025_t=        1,704
   Quinua                                        En fresco                    deptos= 4  prod_2025_t=          550
   Millo                                         En fresco                    deptos= 4  prod_2025_t=          496
## Cultivos para condimentos, bebidas medicinales y aromá

In [33]:
# 32. Desagregaciones que indican destino no alimentario directo (exportacion o
# industria) dentro de cultivos alimentarios. Insumo para la decision de la canasta.
patron_destino = r'export|industrial|az[uú]car'
destino = (df[df['desagregaci_n_cultivo'].str.contains(patron_destino, case=False)]
           .assign(prod=v['producci_n']).groupby(['cultivo', 'desagregaci_n_cultivo', 'a_o'])['prod'].sum().unstack())
destino

,a_o,2024,2025
cultivo,desagregaci_n_cultivo,,
Banano,Banano de exportación,2123715.24,2174248.32
Caña,Caña de azúcar,33048404.87,33007825.70
Plátano,Plátano de exportación,477590.00,441645.30
Yuca,Yuca industrial,292141.20,208537.70
